# **Transformer Model Outline**

A basic transformer based neural network trained on Haruki Murakami's After Dark.

Prerequisite knowlege required. See [Spoticore](../../../spoticore/src/spoticore/).


## Load dataset


In [ ]:
BOOK_PATH = "./book.txt"

with open(BOOK_PATH, "r", encoding="utf-8") as bk:
    text = bk.read()
    lines = text.splitlines()

print("total characters in book:\n", len(text))
print("\nfirst 1000 characters:\n", text[:1000])
print("\nfirst 5 lines:\n", lines[:5])

#### Unique characters in dataset


In [ ]:
chars = sorted(set(text))
print("unique characters:", chars)

chars_str = "".join(chars)
print("\nunique characters as string:", chars_str)

vocab_size = len(chars)
print("\nvocabulary size:", vocab_size)

#### StoI and ItoS mappings


In [ ]:
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

print(f"{stoi = }\n\n{itos = }")

#### encoder and decoder


In [ ]:
import torch


def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]


def decode(encoding: torch.LongTensor) -> str:
    return "".join([itos[encode] for encode in encoding])


ex = "hi there i am vihanga"
print("example:", ex)
enc = encode(ex)
print("\nencoding:", enc)
dec = decode(enc)
print("\ndecoding:", dec)

### Encode book into tensor


In [ ]:
from typing import Final

SEED: Final[int] = 4324

torch.manual_seed(SEED)


data = torch.tensor(encode(text), dtype=torch.long)
print(f"{data.shape = } {data.dtype = }")
print(f"\nfirst 100 characters in model's pov:\n{data[:100]}")

### Split text into train and validation datasets


In [ ]:
lim = int(0.9 * len(data))
train_data = data[:lim]
val_data = data[lim:]

train_data.shape, val_data.shape

In [ ]:
block_size = 8
print("english:", text[: block_size + 1])
print("encoding:", train_data[: block_size + 1])

#### N-1 examples in a sample chunk of size N

This will also help the model to generate predictions from smaller context sizes.


In [ ]:
x = train_data[:block_size]
y = train_data[1 : block_size + 1]

print(f"{x = }]\n{y = }\n")

for i in range(block_size):
    print(f"Input: {x[: i + 1]} --> Target: {y[i]}")

### Generate random training batches


In [ ]:
from enum import Enum


class BatchType(str, Enum):
    train = "data_training_batch"
    val = "data_validation_batch"

In [ ]:
# how many independent sequences processed in parallel
batch_size = 4
# max context length for a prediction
block_size = 8

# * in our case: 1 token --> 1 character


def get_batch(type: BatchType) -> torch.Tensor:
    data = train_data if type == BatchType.train else val_data
    # get batch_size random start indices
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # for each batch, extract block_size consecutive elements from start
    x = torch.stack([data[i : i + block_size] for i in ix])
    # extract targets for each batch
    y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])

    return x, y


xb, yb = get_batch(BatchType.train)

print(f"Inputs shape: {xb.shape}\nOutput shape: {yb.shape}")
print(f"\nInputs: {xb}\nOutputs: {yb}\n")

for i in range(batch_size):  # batch dimension
    for j in range(block_size):  # time dimension
        print(f"Input: {xb[i, j]} --> Target: {yb[i, j]}")

## Baseline: Bigram Language Model


In [ ]:
import torch.nn as nn
from torch.nn import functional as F

### BigramLanguageModel class


#### `nn.Embedding()` is dimension-agnostic for input

Input shape:

$$(\text{d1, d2, d3, ..., dn})$$

Output shape:

$$(\text{d1, d2, d3, ..., dn, }{embedding\ dim})$$

Each scalar integer is replaced with its corresponding embedding vector, adding one extra dimension at the end.

Examples:


In [ ]:
embedding = nn.Embedding(100, 64)

# 1D input - single sequence
idxs = torch.tensor([5, 12, 34, 8])  # shape: (4,)
out = embedding(idxs)  # shape: (4, 64)
print(f"1D output shape: {out.shape}")

# 2D input - batch of sequences
idxs = torch.randint(0, 100, (8, 20))  # shape: (8, 20)
out = embedding(idxs)  # shape: (8, 20, 64)
print(f"2D output shape: {out.shape}")

# 3D input - batch of 2D structures
idxs = torch.randint(0, 100, (4, 10, 10))  # shape: (4, 10, 10)
out = embedding(idxs)  # shape: (4, 10, 10, 64)
print(f"3D output shape: {out.shape}")

#### Reshaping of logits in `forward()`

Before reshaping, the logits tensor has shape (B, T, C) = (4, 8, 81) where:

- B = 4: batch size (4 sequences processed in parallel)
- T = 8: block size (8 time-steps/tokens per sequence)
- C = 81: vocabulary size (81 possible next tokens)

After reshaping, the logits tensor has shape ( 32, 81) where:

- 32 = B × T = 4 × 8: flattened positions (4 sequences × 8 time steps)
- 81: logits for each possible next token

The 4 sequences, each with 8 time steps (tokens) get flattened into 32 positions; resulting in 32 rows of tokens.

For each of these positions/tokens, there are 81 logits representing the unnormalized scores for all possible next tokens.


In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        # The bigram model is essentially a 2D lookup table of logits.
        # This matches the Embedding module where lookups return scores for next tokens.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(
        self, idxs: torch.Tensor, targets: torch.Tensor = None
    ) -> (torch.Tensor, torch.Tensor | None):
        # idxs and targets are both of shape (B, T).
        # Logits resulting from the embedding lookup have shape (B, T, C).
        logits = self.token_embedding_table(idxs)

        # Determine if we are in generation mode (no targets) or training mode.
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # Reshape tensors to (B*T, C) because cross_entropy expects the channel dimension second.
            # This flattens the batch and time dimensions into a single sequence of predictions.
            logits, targets = logits.reshape(B * T, C), targets.reshape(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idxs: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        # Generate new tokens one by one for max_new_tokens steps.
        for _ in range(max_new_tokens):
            # Get predictions for all sequences in the current batch.
            logits, _ = self(idxs)
            # Focus only on the last time-step to predict the next token.
            logits = logits[:, -1, :]  # Shape: (B, C).
            # Apply softmax to convert unnormalized logits into a probability distribution.
            probs = F.softmax(logits, dim=-1)
            # Sample the next token index from the resulting distribution.
            next_idxs = torch.multinomial(probs, num_samples=1)  # Shape: (B, 1).
            # Concatenate the newly sampled token to the existing sequence.
            idxs = torch.cat((idxs, next_idxs), dim=1)  # Shape: (B, T+1).

        return idxs

In [ ]:
bmodel = BigramLanguageModel()

In [ ]:
# 1 batch input
ctx = torch.zeros((1, 1), dtype=torch.long)
preds = bmodel.generate(ctx, max_new_tokens=400)[0].tolist()

# decode predictions
preds_dec = decode(preds)
print("bigram generated output:", preds_dec, "\ntokens:", len(preds_dec))

### Train bigram model


In [ ]:
# Initialize the AdamW optimizer with the model's parameters and a learning rate.
optimizer = torch.optim.AdamW(bmodel.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
iterations = 100_000

for step in range(iterations):
    # Sample a batch of data for training.
    xb, yb = get_batch(BatchType.train)

    # Perform the forward pass to evaluate the loss.
    logits, loss = bmodel(xb, yb)

    # Clear previous gradients to prevent accumulation.
    optimizer.zero_grad()

    # Compute the gradient of the loss with respect to parameters.
    loss.backward()

    # Update the parameters based on the computed gradients.
    optimizer.step()

print(loss.item())

Check quality of generation after training:


In [ ]:
# 1 batch input
ctx = torch.zeros((1, 1), dtype=torch.long)
preds = bmodel.generate(ctx, max_new_tokens=400)[0].tolist()

# decode predictions
preds_dec = decode(preds)
print("bigram generated output:", preds_dec, "\ntokens:", len(preds_dec))

# Appendix


## 1. Optimizers


### a. SGD & SGD+Momentum


Standard SGD (Stochastic Gradient Descent):

$$\theta_{t+1} = \theta_t - \eta \nabla_\theta \mathcal{L}(\theta_t)$$

Where:

- $\theta_t$: parameters at time step $t$.
- $\eta$: learning rate.
- $\nabla_\theta \mathcal{L}(\theta_t)$: gradient of loss with respect to parameters.

**SGD with Momentum:**

$$v_{t+1} = \beta v_t + \nabla_\theta \mathcal{L}(\theta_t)$$

$$\theta_{t+1} = \theta_t - \eta v_{t+1}$$

Where:

- $v_t$: velocity (exponentially weighted average of past gradients).
- $\beta$: momentum coefficient (typically 0.9).
- $v_0 = 0$: initial velocity.

**Key Differences:**

- Standard SGD: Updates parameters directly using current gradient. Can oscillate or get stuck in local minima.
- SGD + Momentum: Accumulates past gradients in velocity term, which smooths updates and accelerates convergence in relevant directions. The momentum term helps escape shallow local minima and reduces oscillations.


### b. AdaGrad


AdaGrad (Adaptive Gradient Algorithm) is an optimization algorithm that adapts the learning rate for each parameter based on the historical gradients for that parameter. It maintains a running sum of squared gradients and uses it to scale the learning rate individually.

$$G_{t+1} = G_t + (\nabla_\theta \mathcal{L}(\theta_t))^2$$

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_{t+1}} + \epsilon} \nabla_\theta \mathcal{L}(\theta_t)$$

Where:

- $G_t$: sum of all squared gradients up to time $t$ (accumulator).
- $\eta$: initial learning rate.
- $\epsilon$: small constant (typically $10^{-8}$) to prevent division by zero.
- $G_0 = 0$: initial accumulator.

Note:

> The accumulator $G_t$ keeps growing since it sums all past squared gradients. This causes the effective learning rate to continuously decrease, eventually becoming infinitesimally small and stopping learning prematurely.


### c. RMSProp


RMSProp (Root Mean Square Propagation) is an adaptive learning rate optimization algorithm that addresses the diminishing learning rates problem in AdaGrad by using an exponentially decaying average of squared gradients.

$$s_{t+1} = \rho s_t + (1 - \rho) (\nabla_\theta \mathcal{L}(\theta_t))^2$$

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{s_{t+1}} + \epsilon} \nabla_\theta \mathcal{L}(\theta_t)$$

Where:

- $s_t$: exponentially weighted average of squared gradients (cache).
- $\rho$: decay rate (typically 0.9 or 0.99).
- $\eta$: learning rate.
- $\epsilon$: small constant (typically $10^{-8}$) to prevent division by zero.
- $s_0 = 0$: initial cache.


#### How RMSProp prevents vanishing learning rates

The key difference between AdaGrad and RMSProp is how they accumulate gradient information:

- AdaGrad: $G_{t+1} = G_t + g_t^2$ [Monotonically increases without bound]
- RMSProp: $s_{t+1} = \rho s_t + (1 - \rho) g_t^2$ [Converges to a stable value]

**Step 1: Understand the update rule**

RMSProp's accumulator update:
$$s_{t+1} = \rho s_t + (1 - \rho) g_t^2$$

where $\rho$ is typically $0.9$ or $0.99$, and $g_t = \nabla_\theta \mathcal{L}(\theta_t)$ is the gradient at time $t$.

**Step 2: Expand the recurrence over multiple timesteps**

Let's expand $s_t$ by substituting previous values:

$$s_1 = \rho \cdot 0 + (1-\rho)g_0^2 = (1-\rho)g_0^2$$

$$s_2 = \rho s_1 + (1-\rho)g_1^2 = \rho(1-\rho)g_0^2 + (1-\rho)g_1^2$$

$$s_3 = \rho s_2 + (1-\rho)g_2^2 = \rho^2(1-\rho)g_0^2 + \rho(1-\rho)g_1^2 + (1-\rho)g_2^2$$

In general:
$$s_t = (1-\rho)\sum_{i=0}^{t-1} \rho^{t-1-i} g_i^2$$

**Step 3: Observe the exponential decay**

Each past gradient $g_i^2$ is weighted by $\rho^{t-1-i}$:

- Recent gradient $g_{t-1}^2$: weight = $(1-\rho) \cdot \rho^0 = (1-\rho)$
- One step back $g_{t-2}^2$: weight = $(1-\rho) \cdot \rho^1$
- Two steps back $g_{t-3}^2$: weight = $(1-\rho) \cdot \rho^2$
- Old gradients decay exponentially: $\rho^k \to 0$ as $k \to \infty$

**Step 4: Analyze steady-state behavior**

Assume gradients have roughly constant magnitude $g$ (i.e., $g_i^2 \approx g^2$ for all $i$). Then:

$$s_t = (1-\rho)g^2 \sum_{i=0}^{t-1} \rho^{t-1-i} = (1-\rho)g^2 \sum_{j=0}^{t-1} \rho^j$$

Using the geometric series formula $\sum_{j=0}^{t-1} \rho^j = \frac{1 - \rho^t}{1 - \rho}$:

$$s_t = (1-\rho)g^2 \cdot \frac{1 - \rho^t}{1 - \rho} = g^2(1 - \rho^t)$$

As $t \to \infty$, since $\rho < 1$, we have $\rho^t \to 0$, so:

$$s_\infty = g^2$$

**Step 5: Compare with AdaGrad**

- AdaGrad: $G_t = \sum_{i=0}^{t-1} g_i^2 \approx t \cdot g^2$ (grows linearly with $t$)
- RMSProp: $s_t \to g^2$ (converges to constant)

**Step 6: Impact on effective learning rate**

The effective learning rate is:

$$\text{Effective LR} = \frac{\eta}{\sqrt{s_t} + \epsilon}$$

- AdaGrad: As $t$ increases, $\sqrt{G_t} \approx \sqrt{t} \cdot g$, so effective LR $\propto \frac{1}{\sqrt{t}} \to 0$
- RMSProp: As $t$ increases, $\sqrt{s_t} \to g$, so effective LR $\approx \frac{\eta}{g}$ (stays constant)


### d. Adam


Adam combines the best of both SGD+Momentum and RMSProp. It maintains two exponentially decaying averages:

1. **First moment** (mean) of gradients - like momentum
2. **Second moment** (uncentered variance) of gradients - like RMSProp

**Algorithm:**

$$m_{t+1} = \beta_1 m_t + (1 - \beta_1) \nabla_\theta \mathcal{L}(\theta_t)$$

$$v_{t+1} = \beta_2 v_t + (1 - \beta_2) (\nabla_\theta \mathcal{L}(\theta_t))^2$$

$$\hat{m}_{t+1} = \frac{m_{t+1}}{1 - \beta_1^{t+1}}$$

$$\hat{v}_{t+1} = \frac{v_{t+1}}{1 - \beta_2^{t+1}}$$

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_{t+1}} + \epsilon} \hat{m}_{t+1}$$

Where:

- $m_t$: first moment estimate (momentum term) - exponentially weighted average of gradients
- $v_t$: second moment estimate - exponentially weighted average of squared gradients (like RMSProp's $s_t$)
- $\beta_1$: decay rate for first moment (typically 0.9)
- $\beta_2$: decay rate for second moment (typically 0.999)
- $\hat{m}_t$, $\hat{v}_t$: bias-corrected moment estimates
- $\eta$: learning rate (typically 0.001)
- $\epsilon$: small constant for numerical stability (typically $10^{-8}$)
- $m_0 = 0$, $v_0 = 0$: initial moments


#### How Adam builds on RMSProp

Adam extends RMSProp by adding momentum and bias correction.

**1. RMSProp Component (Second Moment)**

The RMSProp part of Adam is the second moment estimate:
$$v_{t+1} = \beta_2 v_t + (1 - \beta_2) g_t^2$$

**2. Momentum Component (First Moment)**

Adam adds momentum by tracking the exponentially weighted average of gradients (not squared):
$$m_{t+1} = \beta_1 m_t + (1 - \beta_1) g_t$$

**3. Bias Correction (New Addition)**

Adam introduces bias correction to account for initialization bias ($m_0 = 0$, $v_0 = 0$):

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

**Why bias correction matters:**

In early iterations, $m_t$ and $v_t$ are biased toward zero. For example:

- At $t=1$: $m_1 = (1-\beta_1)g_0$, which is much smaller than $g_0$ if $\beta_1 = 0.9$
- Bias correction: $\hat{m}_1 = \frac{(1-\beta_1)g_0}{1-\beta_1} = g_0$ ✓

As $t \to \infty$, $\beta_1^t \to 0$ and $\beta_2^t \to 0$, so $\hat{m}_t \approx m_t$ and $\hat{v}_t \approx v_t$.

**4. Combined Update Rule**

The final update combines both:
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_{t+1}} + \epsilon} \hat{m}_{t+1}$$

Compare with RMSProp:
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{s_{t+1}} + \epsilon} g_t$$


#### AdamW


AdamW (Adam with Decoupled Weight Decay) is a variant of Adam that fixes the issue with weight decay (L2 regularization) by decoupling it from the gradient-based optimization.

The key difference is that the weight decay is added **after** the adaptive learning rate scaling, not before.

**The Problem with Adam + L2 Regularization:**

In standard Adam, L2 regularization is implemented by adding a penalty term to the loss:
$$\mathcal{L}_{\text{reg}}(\theta) = \mathcal{L}(\theta) + \frac{\lambda}{2} \|\theta\|^2$$

This modifies the gradient:
$$g_t = \nabla_\theta \mathcal{L}(\theta_t) + \lambda \theta_t$$

However, when Adam applies adaptive learning rates, the L2 penalty gets scaled by $\frac{1}{\sqrt{\hat{v}_t}}$, which means:

- Parameters with large gradients get **less** regularization (smaller effective weight decay)
- Parameters with small gradients get **more** regularization (larger effective weight decay)

This is counterintuitive and reduces the effectiveness of regularization.

**AdamW Solution: Decoupled Weight Decay**

AdamW separates weight decay from the gradient computation:

$$m_{t+1} = \beta_1 m_t + (1 - \beta_1) \nabla_\theta \mathcal{L}(\theta_t)$$

$$v_{t+1} = \beta_2 v_t + (1 - \beta_2) (\nabla_\theta \mathcal{L}(\theta_t))^2$$

$$\hat{m}_{t+1} = \frac{m_{t+1}}{1 - \beta_1^{t+1}}, \quad \hat{v}_{t+1} = \frac{v_{t+1}}{1 - \beta_2^{t+1}}$$

$$\theta_{t+1} = \theta_t - \eta \left( \frac{\hat{m}_{t+1}}{\sqrt{\hat{v}_{t+1}} + \epsilon} + \lambda \theta_t \right)$$

Where:

- $m_t$: first moment (exponentially weighted average of gradients) - same as Adam
- $v_t$: second moment (exponentially weighted average of squared gradients) - same as Adam
- $\hat{m}_t$, $\hat{v}_t$: bias-corrected moments - same as Adam
- $\lambda$: weight decay coefficient (typically 0.01 - 0.1)
- Other parameters ($\beta_1$, $\beta_2$, $\eta$, $\epsilon$) have the same meaning as in Adam

| Aspect                   | Adam with L2                                      | AdamW                                  |
| ------------------------ | ------------------------------------------------- | -------------------------------------- |
| Regularization           | $g_t = \nabla\mathcal{L} + \lambda\theta$         | Weight decay separate from gradient    |
| Weight decay scaling     | Scaled by $\frac{1}{\sqrt{\hat{v}_t}}$ (adaptive) | Direct: $\lambda\theta$ (uniform)      |
| Effective regularization | Inconsistent across parameters                    | Consistent across all parameters       |
| Typical $\lambda$        | 0.0001 - 0.001                                    | 0.01 - 0.1 (larger values work better) |


### e. Optimizer Comparison

| Optimizer      | Update Rule                                                                                                                                                        | Key Feature                              | Main Limitation           |
| -------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------ | ---------------------------------------- | ------------------------- |
| SGD            | $\theta \leftarrow \theta - \eta g_t$                                                                                                                              | Simple, good generalization              | Slow, oscillates          |
| SGD + Momentum | $v \leftarrow \beta v + g_t$ <br> $\theta \leftarrow \theta - \eta v$                                                                                              | Accelerates convergence                  | Still uses fixed LR       |
| AdaGrad        | $G \leftarrow G + g_t^2$ <br> $\theta \leftarrow \theta - \frac{\eta}{\sqrt{G}} g_t$                                                                               | Adaptive per-parameter LR                | LR vanishes over time     |
| RMSProp        | $s \leftarrow \rho s + (1-\rho) g_t^2$ <br> $\theta \leftarrow \theta - \frac{\eta}{\sqrt{s}} g_t$                                                                 | Fixes AdaGrad's vanishing LR             | No momentum               |
| Adam           | $m \leftarrow \beta_1 m + (1-\beta_1) g_t$ <br> $v \leftarrow \beta_2 v + (1-\beta_2) g_t^2$ <br> $\theta \leftarrow \theta - \frac{\eta}{\sqrt{\hat{v}}} \hat{m}$ | Momentum + Adaptive LR + Bias correction | May overfit on some tasks |
| AdamW          | Same as Adam but: <br> $\theta \leftarrow \theta - \eta \left(\frac{\hat{m}}{\sqrt{\hat{v}}} + \lambda\theta\right)$                                               | Adam + Proper weight decay               | Requires tuning $\lambda$ |
